# 프라이빗 RAG 구현하기 : 근로기준법 버전 (마무리 과제)

레슨의 완전 로컬 RAG 파이프라인(청킹 비교 → 오픈소스 임베딩 비교 → Qdrant → 로컬 LLM → 선택 심화 트랙)을 **근로기준법**에 그대로 적용하고, 기준선을 만든 뒤 개선 실험을 진행합니다.

**레슨 노트북 대비 바뀐 점**

- 실습 문서 : 가상의 사내 규정(11개 조항) → 근로기준법 본칙 131개 조항 (시행 2026. 8. 20. 기준 현행 조문)
- 문서 정제 단계 추가 : 웹 복사본의 버튼 문구, 시행예정 개정본 중복, 부칙, 삭제 조문을 걸러냅니다 (Step 1)
- 평가 세트 : 정답을 청크 번호 대신 **조번호 문자열**(예: `제60조`)로 적고 번호로 변환합니다. 의미 질문 14개 + 식별자 질문 6개
- 모든 빈칸(Step 4 필터 실습, Step 6 조기 반환, 트랙 1~5)을 채웠습니다
- 트랙 2 : BM25 토크나이저가 `제76조의2` 같은 "조의N" 식별자를 통짜 토큰으로 인식하도록 수정하고, 수정 전후를 비교합니다
- Step 6 오프라인 검증 : 환경변수 설정에 더해 **소켓 연결 자체를 차단**한 상태에서 파이프라인을 실행합니다 (환경변수는 라이브러리를 불러온 뒤에 바꾸면 적용되지 않는 경우가 있기 때문입니다)
- 모든 실험 결과가 `RESULTS`에 자동으로 쌓이고, Step 8에서 기준선 대비 표로 출력됩니다
- VRAM 관리 : 다 쓴 모델(시맨틱 청킹용 e5-small, 트랙 1 처리량 측정용 BGE-M3 사본)은 바로 내립니다

**실행 환경 (Kaggle)**

- Accelerator : GPU T4 x2 (첫 셀에서 한 장만 쓰도록 고정합니다)
- Internet : On
- 문서 : 오른쪽 **Add Input → Upload** 로 `근로기준법_09_29.txt` 를 올립니다. 경로는 코드가 자동으로 찾습니다

## Step 0 : 설치와 준비

필요한 라이브러리를 설치합니다. 이 노트북은 GPU 런타임(T4, VRAM 약 15GB)에서 실행합니다. API 키 등록 단계는 없습니다. 모든 것이 로컬에서 동작합니다.

In [1]:
# Kaggle T4 x2 환경에서 GPU 한 장만 쓰도록 고정합니다 (torch import 전에 실행해야 합니다)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 라이브러리 버전을 지정해 설치합니다
!pip install -q "langchain==0.3.*" "langchain-community==0.3.*" "langchain-text-splitters==0.3.*" \
                "langchain-experimental==0.3.*" "langchain-huggingface==0.3.*"
!pip install -q "sentence-transformers==5.7.*" "qdrant-client==1.12.*"
!pip install -q "transformers==4.57.*" "accelerate==1.14.*" "bitsandbytes==0.50.*"
!pip install -q "rank_bm25==0.2.2"   # 트랙 2(하이브리드 검색)용. 레슨에서는 트랙 2에 있던 설치를 앞으로 옮겼습니다

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, 

설치 중 빨간 글씨의 pip 의존성 경고(`pip's dependency resolver …`)는 Kaggle에 미리 깔린 다른 패키지와의 충돌 안내입니다. 오류가 아니므로 다음 셀이 정상 실행되면 그대로 진행합니다.

In [2]:
import gc, os, re, time
import numpy as np
import pandas as pd
import torch
import transformers, sentence_transformers, qdrant_client

# GPU 확인. T4, VRAM 약 15~16GB로 표시되면 정상입니다
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"transformers {transformers.__version__} / sentence-transformers {sentence_transformers.__version__}")

GPU : Tesla T4
VRAM : 15.6 GB
transformers 4.57.6 / sentence-transformers 5.7.0


## Step 1 : 실습 문서 준비

오늘의 문서는 **근로기준법**입니다. 레슨의 사내 규정과 같은 장·조 구조를 가지면서, 조항 수가 131개로 훨씬 많아 검색 개선 기법의 효과를 확인하기 좋습니다. 법령은 저작권 보호 대상이 아니므로 원문을 그대로 사용할 수 있습니다.

국가법령정보센터 웹 페이지를 복사한 원문에는 검색에 방해가 되는 요소가 섞여 있습니다.

- 조문마다 붙은 버튼 문구 (`조문체계도버튼연혁생활법령버튼` 등)
- **시행예정 개정본**이 현행 조문 바로 뒤에 같은 조번호로 한 번 더 들어 있습니다 (예: 제101조, 제105조, 제116조). 현행 조문은 줄 앞에 공백이 한 칸 있고, 개정본은 공백 없이 시작합니다
- 개정 이력 표기 (`<개정 2019. 1. 15.>`, `[전문개정 …]`)
- 본칙 뒤의 **부칙** : 부칙에도 `제1조`, `제2조`가 있어서 그대로 넣으면 식별자 검색이 본칙과 부칙을 구분하지 못합니다
- 삭제된 조문 (제35조 등)

정제 규칙은 "지금 시행 중인 본칙 조문만, 본문만 남긴다"입니다. 시행예정 개정본은 아직 효력이 없으므로 제외합니다.

In [3]:
import glob

# Kaggle : 오른쪽 Add Input → Upload 로 올린 txt는 /kaggle/input/<데이터셋 이름>/ 아래에 있습니다
# 데이터셋 이름을 몰라도 찾을 수 있게 파일명으로 검색합니다
candidates = (glob.glob("/kaggle/input/**/*근로기준법*.txt", recursive=True)
              + glob.glob("./*근로기준법*.txt"))
if not candidates:   # 업로드 과정에서 파일명이 바뀐 경우 txt 파일 전체에서 찾습니다
    candidates = glob.glob("/kaggle/input/**/*.txt", recursive=True)
assert candidates, "근로기준법 txt 파일을 찾지 못했습니다. Add Input 으로 업로드했는지 확인하세요"

LAW_PATH = candidates[0]
print(f"사용할 파일 : {LAW_PATH}")

사용할 파일 : /kaggle/input/datasets/rolledracoon/rmsfhrlwnsqjq/rmsfhrlwnsqjq 09.29.txt


In [4]:
def load_law(path):
    """근로기준법 웹 복사본에서 현행 본칙 조문만 뽑아 [{조번호, 제목, 장, text}, ...] 로 반환합니다"""
    raw = open(path, encoding="utf-8").read().replace("\r\n", "\n")

    # 1) 부칙 제거 : 첫 번째 "부칙 <법률 …>" 줄부터 끝까지 버립니다
    m = re.search(r"^부칙\s*<", raw, re.M)
    body = raw[:m.start()] if m else raw

    articles, chapter, cur = [], "", None
    for line in body.split("\n"):
        # 2) 장 제목 : 들여쓰기된 줄만 현행입니다 (들여쓰기 없는 장 제목은 시행예정 개정본)
        mc = re.match(r"^\s{2,}(제\d+장(?:의\d+)?\s+[^<]+?)\s*(?:<.*)?$", line)
        if mc:
            chapter = mc.group(1).strip()
            continue

        # 3) 현행 조문 시작 : 공백 한 칸 + 제N조(의M)(제목)
        ma = re.match(r"^ (제\d+조(?:의\d+)?)\(([^)]*)\)\s*(.*)$", line)
        if ma:
            cur = {"조번호": ma.group(1), "제목": ma.group(2), "장": chapter,
                   "lines": [ma.group(3)], "stop": False}
            articles.append(cur)
            continue
        if cur is None:
            continue

        s = line.strip()
        # 4) 다른 조의 "삭제" 표시(예: 제35조 삭제)가 나오면 현재 조문을 닫습니다
        if re.match(r"^제\d+조(의\d+)?\s*삭제", s) and not s.startswith(cur["조번호"]):
            cur = None
            continue
        # 5) 같은 조번호가 다시 나오면 시행예정 개정본이므로 거기서부터 버립니다
        if re.match(rf"^{re.escape(cur['조번호'])}(\(|\s*삭제)", s):
            cur["stop"] = True
        if cur["stop"]:
            continue
        # 6) 빈 줄, 버튼 문구, 시행일 표기는 건너뜁니다
        if not s or "버튼" in s or s.startswith("[시행일"):
            continue
        cur["lines"].append(s)

    out = []
    for a in articles:
        text = "\n".join(a["lines"])
        text = re.sub(r"\s*<(개정|신설)[^>]*>", "", text)                                   # <개정 …>, <신설 …>
        text = re.sub(r"^\[.*\]\s*$", "", text, flags=re.M)                                  # [전문개정 …] 같은 부가 표기 줄
        text = re.sub(r"^(?:[①-⑳]|\d+\.)\s*삭제\s*<[^>]*>\s*$", "", text, flags=re.M)        # 삭제된 항·호
        text = re.sub(r"\n{2,}", "\n", text).strip()
        out.append({"조번호": a["조번호"], "제목": a["제목"], "장": a["장"], "text": text})
    return out


articles = load_law(LAW_PATH)
lengths = [len(a["text"]) for a in articles]

print(f"현행 조문 수 : {len(articles)}개")
print(f"조문 길이   : 평균 {np.mean(lengths):.0f}자, 중앙값 {np.median(lengths):.0f}자, 최대 {max(lengths)}자")
print(f"장 목록     : {list(dict.fromkeys(a['장'] for a in articles))}")

현행 조문 수 : 131개
조문 길이   : 평균 267자, 중앙값 150자, 최대 1948자
장 목록     : ['제1장 총칙', '제2장 근로계약', '제3장 임금', '제4장 근로시간과 휴식', '제5장 여성과 소년', '제6장 안전과 보건', '제6장의2 직장 내 괴롭힘의 금지', '제7장 기능 습득', '제8장 재해보상', '제9장 취업규칙', '제10장 기숙사', '제11장 근로감독관 등', '제12장 벌칙']


In [5]:
# 정제가 잘 됐는지 확인합니다
# 제101조는 현행본만 남아야 하고(시행예정 "노동감독관" 조문이 없어야 함), 제34조 끝에 "제35조 삭제"가 붙으면 안 됩니다
for a in articles:
    if a["조번호"] in ("제34조", "제60조", "제101조"):
        print(f"[{a['장']}] {a['조번호']}({a['제목']})")
        print(a["text"][:300])
        print("-" * 60)

[제2장 근로계약] 제34조(퇴직급여 제도)
사용자가 퇴직하는 근로자에게 지급하는 퇴직급여 제도에 관하여는 「근로자퇴직급여 보장법」이 정하는 대로 따른다.
------------------------------------------------------------
[제4장 근로시간과 휴식] 제60조(연차 유급휴가)
① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.
② 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급휴가를 주어야 한다.
④ 사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2년에 대하여 1일을 가산한 유급휴가를 주어야 한다. 이 경우 가산휴가를 포함한 총 휴가 일수는 25일을 한도로 한다.
⑤ 사용자는 제1항부터 제4항까지의 규정에 따른 휴가를 근로자가 
------------------------------------------------------------
[제11장 근로감독관 등] 제101조(감독 기관)
① 근로조건의 기준을 확보하기 위하여 고용노동부와 그 소속 기관에 근로감독관을 둔다.
② 근로감독관의 자격, 임면(任免), 직무 배치에 관한 사항은 대통령령으로 정한다.
------------------------------------------------------------


정제한 조문을 레슨 문서와 같은 **마크다운 헤더 구조**로 바꿉니다. `# 법령명 / ## 장 / ### 조번호(제목)` 형태로 만들어 두면 Step 2의 구조 기반 청킹(`MarkdownHeaderTextSplitter`)을 레슨 코드 그대로 쓸 수 있습니다.

In [6]:
def to_markdown(articles, title="근로기준법"):
    parts, current_chapter = [f"# {title}"], None
    for a in articles:
        if a["장"] != current_chapter:
            current_chapter = a["장"]
            parts.append(f"## {current_chapter}")
        parts.append(f"### {a['조번호']}({a['제목']})\n{a['text']}")
    return "\n\n".join(parts) + "\n"


document = to_markdown(articles)
with open("labor_law.md", "w", encoding="utf-8") as f:
    f.write(document)

print(f"문서 길이 : {len(document)}자")
print(document[:500])

문서 길이 : 37987자
# 근로기준법

## 제1장 총칙

### 제1조(목적)
이 법은 헌법에 따라 근로조건의 기준을 정함으로써 근로자의 기본적 생활을 보장, 향상시키며 균형 있는 국민경제의 발전을 꾀하는 것을 목적으로 한다.

### 제2조(정의)
① 이 법에서 사용하는 용어의 뜻은 다음과 같다.
1. “근로자”란 직업의 종류와 관계없이 임금을 목적으로 사업이나 사업장에 근로를 제공하는 사람을 말한다.
2. “사용자”란 사업주 또는 사업 경영 담당자, 그 밖에 근로자에 관한 사항에 대하여 사업주를 위하여 행위하는 자를 말한다.
3. “근로”란 정신노동과 육체노동을 말한다.
4. “근로계약”이란 근로자가 사용자에게 근로를 제공하고 사용자는 이에 대하여 임금을 지급하는 것을 목적으로 체결된 계약을 말한다.
5. “임금”이란 사용자가 근로의 대가로 근로자에게 임금, 봉급, 그 밖에 어떠한 명칭으로든지 지급하는 모든 금품을 말한다.
6. “평균임금”이란 이를 산정하여야 할 사유가 발생한 날 이전 3개월 동안에


## Step 2 : 청킹 전략 비교

레슨의 네 가지 청킹 전략을 근로기준법에 적용해 비교합니다.

#### 고정 크기 청킹

In [7]:
from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    separator="",        # 구분자 없이 글자 수로만 자릅니다
    chunk_size=300,
    chunk_overlap=50,
)
fixed_chunks = fixed_splitter.split_text(document)
print(f"청크 수 : {len(fixed_chunks)}")

청크 수 : 152


In [8]:
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"[청크 {i}] ({len(chunk)}자)")
    print(chunk)
    print("-" * 50)

[청크 0] (300자)
# 근로기준법

## 제1장 총칙

### 제1조(목적)
이 법은 헌법에 따라 근로조건의 기준을 정함으로써 근로자의 기본적 생활을 보장, 향상시키며 균형 있는 국민경제의 발전을 꾀하는 것을 목적으로 한다.

### 제2조(정의)
① 이 법에서 사용하는 용어의 뜻은 다음과 같다.
1. “근로자”란 직업의 종류와 관계없이 임금을 목적으로 사업이나 사업장에 근로를 제공하는 사람을 말한다.
2. “사용자”란 사업주 또는 사업 경영 담당자, 그 밖에 근로자에 관한 사항에 대하여 사업주를 위하여 행위하는 자를 말한다.
3. “근로”란 정신노동
--------------------------------------------------
[청크 1] (300자)
로자에 관한 사항에 대하여 사업주를 위하여 행위하는 자를 말한다.
3. “근로”란 정신노동과 육체노동을 말한다.
4. “근로계약”이란 근로자가 사용자에게 근로를 제공하고 사용자는 이에 대하여 임금을 지급하는 것을 목적으로 체결된 계약을 말한다.
5. “임금”이란 사용자가 근로의 대가로 근로자에게 임금, 봉급, 그 밖에 어떠한 명칭으로든지 지급하는 모든 금품을 말한다.
6. “평균임금”이란 이를 산정하여야 할 사유가 발생한 날 이전 3개월 동안에 그 근로자에게 지급된 임금의 총액을 그 기간의 총일수로 나눈 금액을 말한다. 근로자가 취
--------------------------------------------------
[청크 2] (299자)
그 근로자에게 지급된 임금의 총액을 그 기간의 총일수로 나눈 금액을 말한다. 근로자가 취업한 후 3개월 미만인 경우도 이에 준한다.
7. “1주”란 휴일을 포함한 7일을 말한다.
8. “소정(所定)근로시간”이란 제50조, 제69조 본문 또는 「산업안전보건법」 제139조제1항에 따른 근로시간의 범위에서 근로자와 사용자 사이에 정한 근로시간을 말한다.
9. “단시간근로자”란 1주 동안의 소정근로시간이 그 사업장에서 같은 종류의 업무에 종사하는 통상 근로자

#### 재귀적 청킹

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=300,
    chunk_overlap=50,
)
recursive_chunks = recursive_splitter.split_text(document)
print(f"청크 수 : {len(recursive_chunks)}")

for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"[청크 {i}] ({len(chunk)}자)")
    print(chunk)
    print("-" * 50)

청크 수 : 183
[청크 0] (114자)
# 근로기준법

## 제1장 총칙

### 제1조(목적)
이 법은 헌법에 따라 근로조건의 기준을 정함으로써 근로자의 기본적 생활을 보장, 향상시키며 균형 있는 국민경제의 발전을 꾀하는 것을 목적으로 한다.
--------------------------------------------------
[청크 1] (270자)
### 제2조(정의)
① 이 법에서 사용하는 용어의 뜻은 다음과 같다.
1. “근로자”란 직업의 종류와 관계없이 임금을 목적으로 사업이나 사업장에 근로를 제공하는 사람을 말한다.
2. “사용자”란 사업주 또는 사업 경영 담당자, 그 밖에 근로자에 관한 사항에 대하여 사업주를 위하여 행위하는 자를 말한다.
3. “근로”란 정신노동과 육체노동을 말한다.
4. “근로계약”이란 근로자가 사용자에게 근로를 제공하고 사용자는 이에 대하여 임금을 지급하는 것을 목적으로 체결된 계약을 말한다.
--------------------------------------------------
[청크 2] (213자)
5. “임금”이란 사용자가 근로의 대가로 근로자에게 임금, 봉급, 그 밖에 어떠한 명칭으로든지 지급하는 모든 금품을 말한다.
6. “평균임금”이란 이를 산정하여야 할 사유가 발생한 날 이전 3개월 동안에 그 근로자에게 지급된 임금의 총액을 그 기간의 총일수로 나눈 금액을 말한다. 근로자가 취업한 후 3개월 미만인 경우도 이에 준한다.
7. “1주”란 휴일을 포함한 7일을 말한다.
--------------------------------------------------


In [10]:
# 실습 : chunk_size 와 chunk_overlap 조합별로 청크 수와 평균 길이를 비교합니다
for size, overlap in [(150, 0), (300, 50), (500, 100), (800, 150)]:
    my_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=size,
        chunk_overlap=overlap,
    )
    my_chunks = my_splitter.split_text(document)
    print(f"({size:>3}, {overlap:>3}) → 청크 수 {len(my_chunks):>4}, 평균 길이 {np.mean([len(c) for c in my_chunks]):.0f}자")

(150,   0) → 청크 수  402, 평균 길이 93자
(300,  50) → 청크 수  183, 평균 길이 210자
(500, 100) → 청크 수  110, 평균 길이 362자
(800, 150) → 청크 수   68, 평균 길이 587자


근로기준법 조문은 중앙값이 약 150자지만 1,000자가 넘는 조문도 있습니다. 글자 수 기준 청킹은 짧은 조문 여러 개를 한 청크에 섞거나 긴 조문을 여러 조각으로 자르게 됩니다.

#### 문서 구조 기반 청킹

In [11]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "문서"),
    ("##", "장"),
    ("###", "조항"),
]
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_chunks = md_splitter.split_text(document)

print(f"청크 수 : {len(md_chunks)}  (조문 수 {len(articles)}개와 같아야 정상)")
for chunk in md_chunks[:3]:
    print(f"[메타데이터] {chunk.metadata}")
    print(chunk.page_content[:120])
    print("-" * 50)

청크 수 : 131  (조문 수 131개와 같아야 정상)
[메타데이터] {'문서': '근로기준법', '장': '제1장 총칙', '조항': '제1조(목적)'}
이 법은 헌법에 따라 근로조건의 기준을 정함으로써 근로자의 기본적 생활을 보장, 향상시키며 균형 있는 국민경제의 발전을 꾀하는 것을 목적으로 한다.
--------------------------------------------------
[메타데이터] {'문서': '근로기준법', '장': '제1장 총칙', '조항': '제2조(정의)'}
① 이 법에서 사용하는 용어의 뜻은 다음과 같다.
1. “근로자”란 직업의 종류와 관계없이 임금을 목적으로 사업이나 사업장에 근로를 제공하는 사람을 말한다.
2. “사용자”란 사업주 또는 사업 경영 담당자, 그 밖에
--------------------------------------------------
[메타데이터] {'문서': '근로기준법', '장': '제1장 총칙', '조항': '제3조(근로조건의 기준)'}
이 법에서 정하는 근로조건은 최저기준이므로 근로 관계 당사자는 이 기준을 이유로 근로조건을 낮출 수 없다.
--------------------------------------------------


#### 글자 수가 아니라 토큰 수

임베딩 모델의 입력 상한은 토큰 기준이고, 넘겨도 **에러 없이 잘립니다.** 131개 조문 전체의 토큰 수를 재서 상한을 넘는 조문을 확인합니다.

In [12]:
from transformers import AutoTokenizer

tok_e5 = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
tok_bge = AutoTokenizer.from_pretrained("BAAI/bge-m3")

token_rows = []
for c in md_chunks:
    text = c.page_content
    token_rows.append({
        "조항": c.metadata.get("조항", ""),
        "글자": len(text),
        "E5 토큰": len(tok_e5.encode(text)),
        "BGE 토큰": len(tok_bge.encode(text)),
    })
token_df = pd.DataFrame(token_rows)

print(token_df[["글자", "E5 토큰", "BGE 토큰"]].describe().round(0))
over = token_df[token_df["E5 토큰"] > 512]
print(f"\nE5 상한(512)을 넘는 조문 : {len(over)}개")
print(over.to_string(index=False))

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (578 > 512). Running this sequence through the model will result in indexing errors


           글자   E5 토큰  BGE 토큰
count   131.0   131.0   131.0
mean    267.0   149.0   149.0
std     294.0   161.0   161.0
min      25.0    16.0    16.0
25%      85.0    49.0    49.0
50%     150.0    82.0    82.0
75%     312.0   184.0   184.0
max    1948.0  1083.0  1083.0

E5 상한(512)을 넘는 조문 : 5개
                               조항   글자  E5 토큰  BGE 토큰
   제43조의4(상습체불사업주에 대한 보조ㆍ지원 제한 등) 1022    578     578
제44조의3(건설업의 공사도급에 있어서의 임금에 관한 특례) 1948   1083    1083
      제51조의2(3개월을 초과하는 탄력적 근로시간제) 1000    549     549
             제61조(연차 유급휴가의 사용 촉진) 1167    620     620
                    제74조(임산부의 보호) 1329    706     706


위의 `Token indices sequence length is longer than …` 경고는 토큰 수를 세기만 하는 단계라 무시해도 됩니다. 중요한 것은 표입니다. 레슨 문서는 512를 넘는 조항이 없었지만, 근로기준법에는 E5가 뒷부분을 못 보는 조문이 실제로 있습니다. 트랙 3에서 이 영향을 직접 확인합니다.

#### 시맨틱 청킹

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

# 가벼운 다국어 임베딩 모델 (약 470MB)
embeddings_for_chunking = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cuda"},
)
semantic_splitter = SemanticChunker(
    embeddings_for_chunking,
    breakpoint_threshold_type="percentile",
)
semantic_chunks = semantic_splitter.split_text(document)
print(f"청크 수 : {len(semantic_chunks)}")

# 시맨틱 청킹이 끝났으므로 이 모델은 GPU에서 내립니다 (레슨 대비 추가, VRAM 약 0.5GB 확보)
del semantic_splitter, embeddings_for_chunking
gc.collect()
torch.cuda.empty_cache()

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

청크 수 : 26


#### 네 전략 나란히 비교

In [14]:
strategies = {
    "고정 크기": fixed_chunks,
    "재귀적": recursive_chunks,
    "구조 기반": [c.page_content for c in md_chunks],
    "시맨틱": semantic_chunks,
}

print(f"{'전략':<8} {'청크 수':>6} {'평균 길이':>8} {'최소':>6} {'최대':>6}")
print("-" * 42)
for name, chunks_ in strategies.items():
    lengths_ = [len(c) for c in chunks_]
    print(f"{name:<8} {len(chunks_):>6} {np.mean(lengths_):>8.0f} {min(lengths_):>6} {max(lengths_):>6}")

전략         청크 수    평균 길이     최소     최대
------------------------------------------
고정 크기       152      299    236    300
재귀적         183      210     23    298
구조 기반       131      267     25   1948
시맨틱          26     1456     17   7400


In [15]:
# 정성 비교 : "연차는 며칠?"의 근거인 제60조 첫 문장이 각 전략에서 어떤 청크로 존재하는지 확인합니다
for name, chunks_ in strategies.items():
    print(f"=== {name} ===")
    hits = [c for c in chunks_ if "15일의 유급휴가" in c]
    if hits:
        print(hits[0][:200])
    print()

=== 고정 크기 ===
송관련 서비스업
5. 보건업
② 제1항의 경우 사용자는 근로일 종료 후 다음 근로일 개시 전까지 근로자에게 연속하여 11시간 이상의 휴식 시간을 주어야 한다.
위임행정규칙

### 제60조(연차 유급휴가)
① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.
② 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년

=== 재귀적 ===
### 제60조(연차 유급휴가)
① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.
② 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급휴가를 주어야 한다.
④ 사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초

=== 구조 기반 ===
① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.
② 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또는 1년간 80퍼센트 미만 출근한 근로자에게 1개월 개근 시 1일의 유급휴가를 주어야 한다.
④ 사용자는 3년 이상 계속하여 근로한 근로자에게는 제1항에 따른 휴가에 최초 1년을 초과하는 계속 근로 연수 매 2년에

=== 시맨틱 ===
기타 운송관련 서비스업
5. 보건업
② 제1항의 경우 사용자는 근로일 종료 후 다음 근로일 개시 전까지 근로자에게 연속하여 11시간 이상의 휴식 시간을 주어야 한다. 위임행정규칙

### 제60조(연차 유급휴가)
① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다. ② 사용자는 계속하여 근로한 기간이 1년 미만인 근로자 또



구조 기반 청킹에서는 조문 하나가 정확히 청크 하나가 되고, 장·조항 메타데이터가 붙습니다. 법령은 **조가 곧 의미 단위이자 인용 단위**("제60조에 따르면…")이므로, 이후 단계에서는 구조 기반 청킹 결과를 사용합니다.

## Step 3 : 오픈소스 임베딩 비교 실험

평가 세트를 만듭니다. 레슨은 정답을 청크 번호(`6`, `4`…)로 적었지만, 청크가 131개면 번호를 손으로 맞추기 어렵고 틀리기 쉽습니다. 정답을 **조번호 문자열**로 적고 코드가 번호로 바꾸도록 합니다.

질문은 두 종류를 섞습니다.

- **의미 질문** : 법조문의 표현을 일부러 피한 일상 표현 ("밤 10시 넘어서 일하면 돈 더 받아?")
- **식별자 질문** : 조번호로 묻는 질문 ("제76조의2가 뭐야?")

In [16]:
chunks = md_chunks
chunk_texts = [c.page_content for c in chunks]

# 조번호("제60조", "제76조의2")를 청크 번호로 바꾸는 표를 만듭니다
article_ids = [re.match(r"제\d+조(?:의\d+)?", c.metadata["조항"]).group() for c in chunks]
id2idx = {a: i for i, a in enumerate(article_ids)}
assert len(id2idx) == len(chunks), "조번호가 중복된 청크가 있습니다"

for i in list(range(5)) + list(range(len(chunks) - 3, len(chunks))):
    print(f"[{i:>3}] {chunks[i].metadata['장']} / {chunks[i].metadata['조항']}")

[  0] 제1장 총칙 / 제1조(목적)
[  1] 제1장 총칙 / 제2조(정의)
[  2] 제1장 총칙 / 제3조(근로조건의 기준)
[  3] 제1장 총칙 / 제4조(근로조건의 결정)
[  4] 제1장 총칙 / 제5조(근로조건의 준수)
[128] 제12장 벌칙 / 제114조(벌칙)
[129] 제12장 벌칙 / 제115조(양벌규정)
[130] 제12장 벌칙 / 제116조(과태료)


In [17]:
# 평가 세트 : (질문, 정답 조번호)
semantic_questions = [
    ("1년 일하면 연차 며칠 받을 수 있어?",                      "제60조"),     # 연차 유급휴가
    ("4시간 일하면 쉬는 시간은 얼마나 줘야 해?",                "제54조"),     # 휴게
    ("밤 10시 넘어서 일하면 돈 더 받아?",                       "제56조"),     # 연장·야간 및 휴일 근로
    ("회사가 자르려면 며칠 전에 말해줘야 해?",                  "제26조"),     # 해고의 예고
    ("해고를 말로만 통보해도 효력이 있어?",                     "제27조"),     # 해고사유 등의 서면통지
    ("그만두면 밀린 월급은 언제까지 받아야 해?",                "제36조"),     # 금품 청산
    ("월급을 상품권으로 줘도 돼?",                              "제43조"),     # 임금 지급
    ("상사가 지위를 이용해서 괴롭히는 것도 법에 걸려?",         "제76조의2"),  # 직장 내 괴롭힘의 금지
    ("몇 살부터 알바할 수 있어?",                               "제64조"),     # 최저 연령과 취직인허증
    ("출산휴가는 며칠 쓸 수 있어?",                             "제74조"),     # 임산부의 보호
    ("일주일에 기본으로 몇 시간까지 일할 수 있어?",             "제50조"),     # 근로시간
    ("그만두면 위약금 물어내라는 계약서 써도 돼?",              "제20조"),     # 위약 예정의 금지
    ("직원 4명인 가게도 이 법이 적용돼?",                       "제11조"),     # 적용 범위
    ("근로계약서에 꼭 들어가야 하는 내용이 뭐야?",              "제17조"),     # 근로조건의 명시
]
identifier_questions = [
    ("제60조 내용 알려줘",        "제60조"),
    ("제76조의2가 뭐야?",         "제76조의2"),
    ("제23조에 뭐라고 써 있어?",  "제23조"),
    ("제94조 설명해줘",           "제94조"),
    ("제36조 요약해줘",           "제36조"),
    ("제76조의3 알려줘",          "제76조의3"),   # 제76조, 제76조의2 와 헷갈리기 쉬운 식별자
]

sem_eval_set = [(q, id2idx[a]) for q, a in semantic_questions]
id_eval_set = [(q, id2idx[a]) for q, a in identifier_questions]
eval_set = sem_eval_set + id_eval_set
print(f"평가 질문 : 의미 {len(sem_eval_set)}개 + 식별자 {len(id_eval_set)}개 = {len(eval_set)}개")

평가 질문 : 의미 14개 + 식별자 6개 = 20개


모든 실험에서 같은 방식으로 Hit@k를 재도록 공통 함수를 만듭니다. `record()`로 기록한 결과는 `RESULTS`에 쌓이고 Step 8에서 표로 정리됩니다.

- `rank_fn(query, k)` : 질문을 받아 상위 k개 청크 번호 목록을 돌려주는 함수

In [18]:
RESULTS = []
BASELINE_NAME = "기준선 : 구조 기반 청킹 + BGE-M3 밀집 검색 (Qdrant)"


def hit_rates(rank_fn, k=3):
    """의미·식별자·전체 질문별 Hit@k"""
    out = {}
    for kind, dataset in [("의미", sem_eval_set), ("식별자", id_eval_set)]:
        out[kind] = sum(gold in [int(i) for i in rank_fn(q, k)] for q, gold in dataset) / len(dataset)
    out["전체"] = (out["의미"] * len(sem_eval_set) + out["식별자"] * len(id_eval_set)) / len(eval_set)
    return out


def record(name, rank_fn, k=3):
    """실험 결과를 RESULTS 에 기록합니다. 같은 이름으로 다시 실행하면 덮어씁니다"""
    t0 = time.time()
    r = hit_rates(rank_fn, k)
    ms = (time.time() - t0) / len(eval_set) * 1000
    row = {"실험": name, "전체": r["전체"], "의미": r["의미"], "식별자": r["식별자"], "ms/질문": ms}
    RESULTS[:] = [x for x in RESULTS if x["실험"] != name] + [row]
    print(f"[기록] {name}")
    print(f"       Hit@{k} 전체 {r['전체']:.0%} / 의미 {r['의미']:.0%} / 식별자 {r['식별자']:.0%},  {ms:.0f} ms/질문")


def vec_rank_fn(embeddings, doc_vecs, query_prefix=""):
    """임베딩 모델 + 미리 계산한 문서 벡터로 rank_fn 을 만듭니다 (numpy 내적)"""
    def rank_fn(query, k=3):
        q = np.array(embeddings.embed_query(query_prefix + query))
        return list(np.argsort(doc_vecs @ q)[::-1][:k])
    return rank_fn

In [19]:
# 두 모델을 로드합니다. BGE-M3는 처음 실행 시 약 2.2GB를 내려받습니다
bge_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
e5_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"현재 VRAM 사용량 : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

현재 VRAM 사용량 : 3.39 GB


In [20]:
# BGE-M3 : 그대로 임베딩합니다
bge_doc_vecs = np.array(bge_embeddings.embed_documents(chunk_texts))

# E5 : 문서에는 passage prefix 를 붙입니다 (E5의 사용 규칙)
e5_doc_vecs = np.array(e5_embeddings.embed_documents([f"passage: {t}" for t in chunk_texts]))

print(f"BGE-M3 문서 벡터 : {bge_doc_vecs.shape}")
print(f"E5 문서 벡터     : {e5_doc_vecs.shape}")

BGE-M3 문서 벡터 : (131, 1024)
E5 문서 벡터     : (131, 768)


In [21]:
def hit_at_k(query_vec, doc_vecs, gold_idx, k=3):
    """정답 청크가 상위 k개 안에 있으면 1, 없으면 0"""
    scores = doc_vecs @ query_vec            # 정규화된 벡터이므로 내적 = 코사인 유사도
    top_k = np.argsort(scores)[::-1][:k]
    return int(gold_idx in top_k), top_k


def evaluate(name, embeddings, doc_vecs, query_prefix=""):
    """레슨의 evaluate 에 의미/식별자 구분과 조번호 표시를 더했습니다"""
    print(f"=== {name} ===")
    total = {"의미": [0, 0], "식별자": [0, 0]}
    for kind, dataset in [("의미", sem_eval_set), ("식별자", id_eval_set)]:
        for q, gold in dataset:
            q_vec = np.array(embeddings.embed_query(query_prefix + q))
            h1, _ = hit_at_k(q_vec, doc_vecs, gold, k=1)
            h3, top3 = hit_at_k(q_vec, doc_vecs, gold, k=3)
            total[kind][0] += h1
            total[kind][1] += h3
            mark = "O" if h3 else "X"
            print(f"  [{mark}] ({kind}) {q}  →  top3 : {[article_ids[i] for i in top3]}, 정답 : {article_ids[gold]}")
    for kind, dataset in [("의미", sem_eval_set), ("식별자", id_eval_set)]:
        n = len(dataset)
        print(f"  {kind:<4} Hit@1 : {total[kind][0]}/{n} ({total[kind][0]/n:.0%})   Hit@3 : {total[kind][1]}/{n} ({total[kind][1]/n:.0%})")
    print()


# E5는 질문에 query prefix 를 붙입니다 (E5의 사용 규칙)
evaluate("BGE-M3", bge_embeddings, bge_doc_vecs)
evaluate("multilingual-e5-base", e5_embeddings, e5_doc_vecs, query_prefix="query: ")

=== BGE-M3 ===
  [O] (의미) 1년 일하면 연차 며칠 받을 수 있어?  →  top3 : ['제60조', '제84조', '제73조'], 정답 : 제60조
  [O] (의미) 4시간 일하면 쉬는 시간은 얼마나 줘야 해?  →  top3 : ['제54조', '제50조', '제52조'], 정답 : 제54조
  [O] (의미) 밤 10시 넘어서 일하면 돈 더 받아?  →  top3 : ['제56조', '제53조', '제70조'], 정답 : 제56조
  [O] (의미) 회사가 자르려면 며칠 전에 말해줘야 해?  →  top3 : ['제26조', '제24조', '제73조'], 정답 : 제26조
  [O] (의미) 해고를 말로만 통보해도 효력이 있어?  →  top3 : ['제27조', '제26조', '제104조'], 정답 : 제27조
  [O] (의미) 그만두면 밀린 월급은 언제까지 받아야 해?  →  top3 : ['제36조', '제37조', '제60조'], 정답 : 제36조
  [O] (의미) 월급을 상품권으로 줘도 돼?  →  top3 : ['제43조', '제38조', '제47조'], 정답 : 제43조
  [O] (의미) 상사가 지위를 이용해서 괴롭히는 것도 법에 걸려?  →  top3 : ['제76조의2', '제116조', '제76조의3'], 정답 : 제76조의2
  [O] (의미) 몇 살부터 알바할 수 있어?  →  top3 : ['제64조', '제70조', '제69조'], 정답 : 제64조
  [O] (의미) 출산휴가는 며칠 쓸 수 있어?  →  top3 : ['제74조', '제60조', '제70조'], 정답 : 제74조
  [O] (의미) 일주일에 기본으로 몇 시간까지 일할 수 있어?  →  top3 : ['제69조', '제50조', '제51조'], 정답 : 제50조
  [O] (의미) 그만두면 위약금 물어내라는 계약서 써도 돼?  →  top3 : ['제19조', '제20조', '제30조'], 정답 : 제20조
  [O] (의미) 직원 4명

In [22]:
# 실습 : E5를 prefix 없이 평가합니다. 문서 벡터도 prefix 없이 다시 만들어야 공정한 비교입니다
e5_doc_vecs_noprefix = np.array(e5_embeddings.embed_documents(chunk_texts))
evaluate("E5 (prefix 없음)", e5_embeddings, e5_doc_vecs_noprefix)

# 임베딩 모델 교체 실험으로 표에 기록합니다
record("임베딩 교체 : multilingual-e5-base (prefix 준수)", vec_rank_fn(e5_embeddings, e5_doc_vecs, "query: "))
record("임베딩 교체 : multilingual-e5-base (prefix 누락)", vec_rank_fn(e5_embeddings, e5_doc_vecs_noprefix))

=== E5 (prefix 없음) ===
  [O] (의미) 1년 일하면 연차 며칠 받을 수 있어?  →  top3 : ['제16조', '제84조', '제60조'], 정답 : 제60조
  [O] (의미) 4시간 일하면 쉬는 시간은 얼마나 줘야 해?  →  top3 : ['제54조', '제50조', '제51조의3'], 정답 : 제54조
  [O] (의미) 밤 10시 넘어서 일하면 돈 더 받아?  →  top3 : ['제56조', '제70조', '제93조'], 정답 : 제56조
  [O] (의미) 회사가 자르려면 며칠 전에 말해줘야 해?  →  top3 : ['제26조', '제24조', '제75조'], 정답 : 제26조
  [O] (의미) 해고를 말로만 통보해도 효력이 있어?  →  top3 : ['제27조', '제26조', '제104조'], 정답 : 제27조
  [X] (의미) 그만두면 밀린 월급은 언제까지 받아야 해?  →  top3 : ['제37조', '제108조', '제45조'], 정답 : 제36조
  [X] (의미) 월급을 상품권으로 줘도 돼?  →  top3 : ['제86조', '제68조', '제64조'], 정답 : 제43조
  [O] (의미) 상사가 지위를 이용해서 괴롭히는 것도 법에 걸려?  →  top3 : ['제76조의2', '제115조', '제109조'], 정답 : 제76조의2
  [O] (의미) 몇 살부터 알바할 수 있어?  →  top3 : ['제64조', '제75조', '제68조'], 정답 : 제64조
  [O] (의미) 출산휴가는 며칠 쓸 수 있어?  →  top3 : ['제74조', '제60조', '제75조'], 정답 : 제74조
  [O] (의미) 일주일에 기본으로 몇 시간까지 일할 수 있어?  →  top3 : ['제69조', '제51조', '제50조'], 정답 : 제50조
  [O] (의미) 그만두면 위약금 물어내라는 계약서 써도 돼?  →  top3 : ['제20조', '제22조', '제68조'], 정답 : 제20조
  [O] 

틀린 문항의 top3를 보면 모델이 무엇과 혼동했는지 읽을 수 있습니다. 특히 식별자 질문을 보세요. 구조 기반 청킹의 `page_content`에는 **조번호가 들어 있지 않습니다**(메타데이터로 분리됨). 그래서 "제60조 알려줘"는 밀집 검색만으로는 맞히기 어렵습니다. 이 한계를 트랙 2에서 다룹니다.

이후 단계에서는 BGE-M3를 사용합니다. (측정 결과 E5가 앞섰다면 E5로 진행해도 됩니다. 그 경우 prefix 규칙과 벡터 차원 768을 잊지 마세요.)

## Step 4 : Qdrant 로컬 세팅

In [23]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

# 이 셀을 다시 실행하면 "폴더가 이미 사용 중" 오류가 나므로, 열려 있는 클라이언트를 먼저 닫습니다 (레슨 대비 추가)
try:
    client.close()
except NameError:
    pass

# path 를 지정하면 임베디드 모드입니다. 해당 폴더에 데이터가 저장됩니다
client = QdrantClient(path="./qdrant_law")
COLLECTION = "labor_law"

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),   # BGE-M3 차원 수
)

# 포인트 id = 청크 번호로 맞춰 두면 평가할 때 바로 비교할 수 있습니다
points = [
    PointStruct(
        id=i,
        vector=bge_doc_vecs[i].tolist(),
        payload={
            "text": chunks[i].page_content,
            "장": chunks[i].metadata.get("장", ""),
            "조항": chunks[i].metadata.get("조항", ""),
            "조번호": article_ids[i],
        },
    )
    for i in range(len(chunks))
]
client.upsert(collection_name=COLLECTION, points=points)
print(f"{len(points)}개 포인트 저장 완료")

131개 포인트 저장 완료


In [24]:
# 기본 검색
query = "1년 일하면 연차 며칠 받을 수 있어?"
results = client.query_points(
    collection_name=COLLECTION,
    query=bge_embeddings.embed_query(query),
    limit=3,
).points

for r in results:
    print(f"[유사도 {r.score:.3f}] {r.payload['조항']}")
    print(f"  {r.payload['text'][:70]}...")

[유사도 0.649] 제60조(연차 유급휴가)
  ① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.
② 사용자는 계속하여 근로한 기간이 1...
[유사도 0.613] 제84조(일시보상)
  제78조에 따라 보상을 받는 근로자가 요양을 시작한 지 2년이 지나도 부상 또는 질병이 완치되지 아니하는 경우에는 사용자는 그...
[유사도 0.590] 제73조(생리휴가)
  사용자는 여성 근로자가 청구하면 월 1일의 생리휴가를 주어야 한다....


In [25]:
# payload 필터 : "제4장 근로시간과 휴식" 안에서만 검색합니다
results = client.query_points(
    collection_name=COLLECTION,
    query=bge_embeddings.embed_query("쉬는 날은 일주일에 몇 번이야?"),
    query_filter=Filter(must=[FieldCondition(key="장", match=MatchValue(value="제4장 근로시간과 휴식"))]),
    limit=3,
).points
for r in results:
    print(f"[유사도 {r.score:.3f}] {r.payload['조항']}")

[유사도 0.580] 제51조의2(3개월을 초과하는 탄력적 근로시간제)
[유사도 0.570] 제59조(근로시간 및 휴게시간의 특례)
[유사도 0.569] 제60조(연차 유급휴가)


In [26]:
# 실습 : "제3장 임금" 범위로 한정해서 "월급은 한 달에 몇 번 줘야 해?"를 검색합니다
results = client.query_points(
    collection_name=COLLECTION,
    query=bge_embeddings.embed_query("월급은 한 달에 몇 번 줘야 해?"),
    query_filter=Filter(must=[FieldCondition(key="장", match=MatchValue(value="제3장 임금"))]),
    limit=3,
).points
for r in results:
    print(f"[유사도 {r.score:.3f}] {r.payload['조항']}")

[유사도 0.656] 제43조(임금 지급)
[유사도 0.572] 제43조의8(체불 임금등에 대한 손해배상청구)
[유사도 0.572] 제44조의2(건설업에서의 임금 지급 연대책임)


In [27]:
# 프라이빗 관점의 확인 : 우리 벡터는 지금 로컬 폴더에 있습니다
!ls -lh ./qdrant_law/collection/labor_law/

total 1.4M
-rw-r--r-- 1 root root 1.4M Sep 29 07:37 storage.sqlite


## Step 5 : 로컬 LLM 로드

생성은 **Qwen2.5-7B-Instruct**가 담당합니다. T4에서 7B를 fp16으로 로드하면 여유가 없으므로 **4bit 양자화**로 로드합니다. T4는 bf16을 지원하지 않으므로 계산 정밀도는 fp16입니다.

In [28]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,       # T4는 bf16 미지원
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)
model.eval()

print(f"모델 로드 완료. VRAM 사용량 : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

모델 로드 완료. VRAM 사용량 : 8.97 GB


In [29]:
def generate(system_prompt: str, user_prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    torch.manual_seed(42)   # 샘플링 결과를 재현할 수 있도록 고정합니다 (레슨 대비 추가)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print(generate("당신은 친절한 비서입니다.", "안녕하세요, 자기소개 해주세요.", max_new_tokens=100))

안녕하세요! 저는 당신의 비서로 활동하는 인공지능 비서입니다. 다양한 정보 제공, 일정 관리, 질문에 대한 답변 등 다양한 업무를 도와드릴 수 있습니다. 항상 고객님의 편리함을 위해 최선을 다할 것입니다. 오늘도 좋은 하루 보내세요!


## Step 6 : 완성 예제, 완전 로컬 RAG

In [30]:
def retrieve(query: str, top_k: int = 3, collection: str = COLLECTION) -> list:
    """로컬 임베딩 + 로컬 Qdrant 검색"""
    results = client.query_points(
        collection_name=collection,
        query=bge_embeddings.embed_query(query),
        limit=top_k,
    ).points
    return [
        {"id": r.id, "조항": r.payload["조항"], "내용": r.payload["text"], "유사도": r.score}
        for r in results
    ]


SYSTEM_PROMPT = """당신은 근로기준법 조문을 안내하는 챗봇입니다.
아래 규칙을 지키세요.
1. 제공된 조문 내용에 근거해서만 답변합니다.
2. 답변 끝에 근거 조항(예: 제60조)을 표기합니다.
3. 제공된 조문에 없는 내용이면 "근로기준법에서 확인되지 않습니다"라고 답합니다.
4. 법률 자문이 아니라 조문 안내임을 전제로, 단정적인 법률 판단은 피합니다."""


def rag_answer(query: str, top_k: int = 3) -> str:
    contexts = retrieve(query, top_k)
    context_text = "\n\n".join(f"[{c['조항']}]\n{c['내용']}" for c in contexts)
    user_prompt = f"""다음은 근로기준법에서 검색된 조문입니다.

{context_text}

질문 : {query}"""
    return generate(SYSTEM_PROMPT, user_prompt)

In [31]:
print(rag_answer("1년 일하면 연차 며칠 받을 수 있어?"))

1년 동안 근로하면 제60조 제1항에 따라 15일의 유급휴가를 받을 수 있습니다. 

근거 조항: 제60조 제1항


In [32]:
questions = [
    "알바도 연차를 받을 수 있어?",
    "야간수당은 얼마나 더 받아?",
    "올해 최저임금은 얼마야?",        # 근로기준법이 아니라 최저임금법에 있는 내용입니다
]
for q in questions:
    print(f"Q : {q}")
    print(f"A : {rag_answer(q)}")
    print("=" * 60)

Q : 알바도 연차를 받을 수 있어?
A : 근로기준법에서 알바도 연차를 받을 수 있는지에 대한 명확한 조문을 찾을 수 없습니다. 따라서 "근로기준법에서 확인되지 않습니다"라고 답합니다.
Q : 야간수당은 얼마나 더 받아?
A : 야간근로에 대한 추가 급여는 통상임금의 100분의 50 이상을 가산해야 합니다. 이는 제56조 제3항에 따라야 합니다.
Q : 올해 최저임금은 얼마야?
A : 근로기준법에서 최저임금的具体数值没有在提供的条文中提及。您可以查阅最新的政府公告或官方网站以获取当前的最低工资标准。


In [33]:
# 실습 : 가장 유사한 조항의 유사도가 기준 미만이면 LLM을 부르지 않고 바로 반환합니다
def rag_answer_v2(query: str, top_k: int = 3, threshold: float = 0.4) -> str:
    contexts = retrieve(query, top_k)

    # 조기 반환 : 가장 유사한 조항조차 기준 미달이면 LLM을 부르지 않습니다
    if contexts[0]["유사도"] < threshold:
        return f"근로기준법에서 확인되지 않습니다. (최고 유사도 {contexts[0]['유사도']:.3f})"

    context_text = "\n\n".join(f"[{c['조항']}]\n{c['내용']}" for c in contexts)
    user_prompt = f"다음은 근로기준법에서 검색된 조문입니다.\n\n{context_text}\n\n질문 : {query}"
    return generate(SYSTEM_PROMPT, user_prompt)


for q in ["올해 최저임금은 얼마야?", "점심 메뉴 추천해줘"]:
    print(f"Q : {q}  (최고 유사도 {retrieve(q, 1)[0]['유사도']:.3f})")
    print(f"A : {rag_answer_v2(q)}")
    print("=" * 60)

Q : 올해 최저임금은 얼마야?  (최고 유사도 0.498)
A : 근로기준법에서 최저임금的具体数值没有在提供的条文中提及。您可以查阅最新的政府公告或官方网站以获取当前的最低工资标准。
Q : 점심 메뉴 추천해줘  (최고 유사도 0.437)
A : 근로기준법에서 점심 메뉴에 대한 내용은 없습니다.


법령은 조문끼리 표현이 비슷해서, 관련 없는 질문도 유사도가 꽤 높게 나올 수 있습니다. 위 출력의 유사도를 보고 threshold가 이 문서에서 적절한지 판단해 보세요.

#### 오프라인 검증

이 파이프라인이 외부와 통신하지 않는지 확인합니다. 레슨은 `HF_HUB_OFFLINE` 환경변수만 켰는데, 이 값은 라이브러리를 **불러올 때** 읽히는 경우가 있어서 실행 도중에 바꾸면 실제로 적용되지 않을 수 있습니다. 그래서 여기서는 파이썬의 **소켓 연결 자체를 막은 상태**에서 파이프라인을 실행합니다. 차단이 실제로 걸렸는지도 먼저 확인합니다.

In [34]:
import socket, urllib.request

os.environ["HF_HUB_OFFLINE"] = "1"          # 레슨과 같은 설정
os.environ["TRANSFORMERS_OFFLINE"] = "1"

_original_connect = socket.socket.connect
def _blocked_connect(self, address):
    raise ConnectionError(f"[차단] 외부 연결 시도 : {address}")

socket.socket.connect = _blocked_connect
try:
    # 1) 차단이 실제로 걸렸는지 확인합니다
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=5)
        print("경고 : 외부 연결이 차단되지 않았습니다")
    except Exception as e:
        print(f"차단 확인 : {type(e).__name__} - {e}")

    # 2) 차단 상태에서 전체 파이프라인(임베딩 → Qdrant 검색 → LLM 생성)을 실행합니다
    print()
    print(rag_answer("출산휴가는 며칠 쓸 수 있어?"))
    print("\n네트워크 차단 상태에서 정상 동작을 확인했습니다")
finally:
    socket.socket.connect = _original_connect   # 검증이 끝나면 원래대로 돌려놓습니다

차단 확인 : URLError - <urlopen error [차단] 외부 연결 시도 : ('2600:9000:2201:de00:17:b174:6d00:93a1', 443, 0, 0)>

임산부의 출산휴가는 출산 전과 출산 후를 통틀어 90일(미숙아를 출산한 경우에는 100일, 한 번에 둘 이상 자녀를 임신한 경우에는 120일)까지 주어야 합니다. 이 중 출산 후의 휴가 기간은 45일(한 번에 둘 이상 자녀를 임신한 경우에는 60일) 이상이어야 합니다.

따라서 출산휴가는 최소 45일에서 최대 100일까지 가능하며, 이는 고용노동부령으로 정하는 사항에 따라 달라질 수 있습니다.

근거 조항: 제74조 제1항

네트워크 차단 상태에서 정상 동작을 확인했습니다


#### 기준선 기록

레슨 파이프라인 그대로(구조 기반 청킹 + BGE-M3 + Qdrant 검색)의 Hit@3을 **기준선**으로 기록합니다. Step 7의 모든 개선 실험은 이 값과 비교합니다.

In [35]:
def rank_baseline(query, k=3):
    return [h["id"] for h in retrieve(query, k)]

record(BASELINE_NAME, rank_baseline)

[기록] 기준선 : 구조 기반 청킹 + BGE-M3 밀집 검색 (Qdrant)
       Hit@3 전체 70% / 의미 100% / 식별자 0%,  24 ms/질문


## Step 7 : 선택 심화 트랙 (다섯 트랙 모두 수행)

In [36]:
# Step 6 에서 켠 오프라인 모드를 해제합니다 (트랙 5의 리랭커를 새로 내려받아야 합니다)
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
print("네트워크 접근을 다시 허용했습니다")

네트워크 접근을 다시 허용했습니다


#### 트랙 1 : 인덱싱 처리량과 배치 최적화

배치 크기별로 소요 시간, 처리량, VRAM 최대 사용량을 잽니다. 근로기준법 청크는 레슨 문서보다 길어서(최대 약 1,000 토큰) 큰 배치에서 메모리가 부족할 수 있습니다. OOM이 나도 멈추지 않고 기록하도록 했습니다.

In [37]:
from sentence_transformers import SentenceTransformer

# batch_size 를 직접 제어하기 위해 sentence-transformers 를 씁니다 (측정 후 바로 내립니다)
st_model = SentenceTransformer("BAAI/bge-m3", device="cuda")

# 1000청크 규모의 코퍼스를 만듭니다 (같은 조문을 사본으로 늘립니다)
large_corpus = [
    f"[사본{i // len(chunk_texts)}] {chunk_texts[i % len(chunk_texts)]}"
    for i in range(1000)
]
print(f"코퍼스 크기 : {len(large_corpus)}청크")

코퍼스 크기 : 1000청크


In [38]:
def measure(batch_size):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()

    torch.cuda.synchronize()
    start = time.time()
    st_model.encode(large_corpus, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)
    torch.cuda.synchronize()                 # GPU 작업이 끝날 때까지 기다려야 시간이 정확합니다
    elapsed = time.time() - start

    throughput = len(large_corpus) / elapsed                     # 초당 처리 청크 수
    peak_vram = torch.cuda.max_memory_allocated() / 1e9          # 이미 올라간 LLM 등을 포함한 전체 최대치
    extra_vram = (torch.cuda.max_memory_allocated() - base) / 1e9  # 인코딩 때문에 늘어난 양
    return elapsed, throughput, peak_vram, extra_vram


throughput_rows = []
print(f"{'배치':>6} {'시간(초)':>10} {'처리량(청크/s)':>16} {'최대 VRAM(GB)':>15} {'증가분(GB)':>12}")
print("-" * 66)
for bs in [1, 8, 32, 64]:
    try:
        e, tp, vram, extra = measure(bs)
        throughput_rows.append({"배치": bs, "시간(초)": e, "처리량(청크/s)": tp, "최대 VRAM(GB)": vram, "증가분(GB)": extra})
        print(f"{bs:>6} {e:>10.1f} {tp:>16.1f} {vram:>15.2f} {extra:>12.2f}")
    except torch.cuda.OutOfMemoryError:
        throughput_rows.append({"배치": bs, "시간(초)": None, "처리량(청크/s)": None, "최대 VRAM(GB)": "OOM", "증가분(GB)": None})
        print(f"{bs:>6}   OOM (메모리 부족)")
        gc.collect()
        torch.cuda.empty_cache()

    배치      시간(초)        처리량(청크/s)     최대 VRAM(GB)      증가분(GB)
------------------------------------------------------------------
     1       42.4             23.6           11.29         0.05
     8       32.1             31.1           11.67         0.43
    32       34.7             28.8           12.96         1.72
    64   OOM (메모리 부족)


In [39]:
# 측정이 끝났으므로 BGE-M3 사본을 내립니다 (VRAM 약 2.2GB 확보, 레슨 대비 추가)
del st_model
gc.collect()
torch.cuda.empty_cache()
print(f"현재 VRAM : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

현재 VRAM : 8.97 GB


배치를 키우면 처리량이 늘다가 어느 지점부터 거의 늘지 않고 VRAM 증가분만 커집니다. 그 꺾이는 지점이 이 GPU(T4)에서의 운영점입니다. 결과는 GPU 종류에 따라 달라지므로 공유할 때 T4 기준임을 함께 적어 주세요.

#### 트랙 2 : 하이브리드 검색 (BM25 + 밀집, RRF)

법령 검색에서 식별자 질문("제60조 알려줘")은 핵심입니다. 두 가지를 준비합니다.

1. **조번호를 본문에 포함** : 구조 기반 청킹의 `page_content`에는 조번호가 없으므로 `장 + 조항 + 본문`을 합친 텍스트를 만듭니다
2. **토크나이저 수정** : 레슨의 `tokenize_ko`는 `제\d+조`만 통짜 토큰으로 잡습니다. 근로기준법에는 `제76조의2`, `제76조의3`처럼 "조의N" 조문이 있어서, 레슨 토크나이저로는 이 셋이 모두 `제76조`로 뭉개집니다. `제\d+조(?:의\d+)?`로 고칩니다

In [40]:
from rank_bm25 import BM25Okapi

full_texts = [
    f"{c.metadata.get('장', '')} {c.metadata.get('조항', '')}\n{c.page_content}"
    for c in chunks
]
print(full_texts[id2idx["제60조"]][:80])


def tokenize_ko_lesson(text):
    """레슨 원본 : 한글/영문/숫자 경계 분리 + 제N조"""
    tokens = re.findall(r"[가-힣]+|[A-Za-z]+|[0-9]+", text)
    tokens += re.findall(r"제\d+조", text)
    return tokens


def tokenize_ko(text):
    """수정본 : 제N조의M 까지 하나의 식별자 토큰으로 인식합니다"""
    tokens = re.findall(r"[가-힣]+|[A-Za-z]+|[0-9]+", text)
    tokens += re.findall(r"제\d+조(?:의\d+)?", text)
    return tokens


print("레슨 토크나이저 :", [t for t in tokenize_ko_lesson("제76조의2가 뭐야?") if t.startswith("제")])
print("수정 토크나이저 :", [t for t in tokenize_ko("제76조의2가 뭐야?") if t.startswith("제")])

bm25_lesson = BM25Okapi([tokenize_ko_lesson(t) for t in full_texts])
bm25 = BM25Okapi([tokenize_ko(t) for t in full_texts])

# 밀집 검색도 같은 텍스트(조번호 포함)로 다시 임베딩해 공정하게 비교합니다
full_vecs = np.array(bge_embeddings.embed_documents(full_texts))

제4장 근로시간과 휴식 제60조(연차 유급휴가)
① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.
② 사용
레슨 토크나이저 : ['제', '제76조']
수정 토크나이저 : ['제', '제76조의2']


In [41]:
def dense_rank(query, top_n=10):
    """밀집 검색(조번호 포함 텍스트) 결과를 청크 번호 순위 리스트로 반환"""
    q = np.array(bge_embeddings.embed_query(query))
    return [int(i) for i in np.argsort(full_vecs @ q)[::-1][:top_n]]


def bm25_rank(query, top_n=10, index=None, tokenizer=None):
    """BM25 검색 결과를 청크 번호 순위 리스트로 반환 (기본값은 수정 토크나이저)"""
    index = index or bm25
    tokenizer = tokenizer or tokenize_ko
    scores = index.get_scores(tokenizer(query))
    return [int(i) for i in np.argsort(scores)[::-1][:top_n]]


def bm25_lesson_rank(query, top_n=10):
    return bm25_rank(query, top_n, index=bm25_lesson, tokenizer=tokenize_ko_lesson)


for q, gold in id_eval_set:
    print(f"Q : {q}  (정답 {article_ids[gold]})")
    print(f"  밀집           top3 : {[article_ids[i] for i in dense_rank(q, 3)]}")
    print(f"  BM25(레슨)     top3 : {[article_ids[i] for i in bm25_lesson_rank(q, 3)]}")
    print(f"  BM25(수정)     top3 : {[article_ids[i] for i in bm25_rank(q, 3)]}")
    print()

Q : 제60조 내용 알려줘  (정답 제60조)
  밀집           top3 : ['제61조', '제60조', '제109조']
  BM25(레슨)     top3 : ['제62조', '제61조', '제18조']
  BM25(수정)     top3 : ['제62조', '제61조', '제18조']

Q : 제76조의2가 뭐야?  (정답 제76조의2)
  밀집           top3 : ['제76조의2', '제76조의3', '제109조']
  BM25(레슨)     top3 : ['제116조', '제76조의2', '제109조']
  BM25(수정)     top3 : ['제76조의2', '제116조', '제109조']

Q : 제23조에 뭐라고 써 있어?  (정답 제23조)
  밀집           top3 : ['제23조', '제22조', '제43조의2']
  BM25(레슨)     top3 : ['제107조', '제23조', '제24조']
  BM25(수정)     top3 : ['제107조', '제23조', '제24조']

Q : 제94조 설명해줘  (정답 제94조)
  밀집           top3 : ['제109조', '제94조', '제9조']
  BM25(레슨)     top3 : ['제94조', '제114조', '제109조']
  BM25(수정)     top3 : ['제94조', '제114조', '제109조']

Q : 제36조 요약해줘  (정답 제36조)
  밀집           top3 : ['제109조', '제76조의3', '제38조']
  BM25(레슨)     top3 : ['제109조', '제36조', '제37조']
  BM25(수정)     top3 : ['제109조', '제36조', '제37조']

Q : 제76조의3 알려줘  (정답 제76조의3)
  밀집           top3 : ['제76조의3', '제109조', '제76조의2']
  BM25(레슨)     top3 : ['제76조', '제76조의2', '제116

In [42]:
# 실습 : RRF로 여러 순위 리스트를 병합합니다
# 각 리스트를 돌면서 문서별로 1/(k + 순위)를 누적합니다. 순위는 0부터 시작하므로 +1 을 해줍니다
def rrf(rank_lists, k=60, top_n=3):
    scores = {}
    for ranks in rank_lists:
        for rank, doc_idx in enumerate(ranks):
            scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (k + rank + 1)
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[:top_n]


def hybrid_rank(query, top_n=3):
    return rrf([dense_rank(query), bm25_rank(query)], top_n=top_n)


print([article_ids[i] for i in hybrid_rank("제76조의2가 뭐야?")])

['제76조의2', '제109조', '제116조']


In [43]:
# 개선 실험으로 기록합니다
record("청킹 변경 : 장·조항 제목을 본문에 포함 (밀집)", dense_rank)
record("BM25 (레슨 토크나이저)", bm25_lesson_rank)
record("BM25 (조의N 대응 토크나이저)", bm25_rank)
record("하이브리드 : 밀집(제목 포함) + BM25(수정), RRF k=60", hybrid_rank)

[기록] 청킹 변경 : 장·조항 제목을 본문에 포함 (밀집)
       Hit@3 전체 90% / 의미 93% / 식별자 83%,  23 ms/질문
[기록] BM25 (레슨 토크나이저)
       Hit@3 전체 55% / 의미 50% / 식별자 67%,  0 ms/질문
[기록] BM25 (조의N 대응 토크나이저)
       Hit@3 전체 55% / 의미 50% / 식별자 67%,  0 ms/질문
[기록] 하이브리드 : 밀집(제목 포함) + BM25(수정), RRF k=60
       Hit@3 전체 85% / 의미 86% / 식별자 83%,  24 ms/질문


식별자 질문 결과를 특히 보세요. 조번호는 **다른 조문의 본문에도 자주 등장합니다**(예: 제17조 본문에 "제60조에 따른 연차 유급휴가"). BM25는 이런 인용도 똑같이 매치하므로, 정답 조문보다 그 조문을 인용한 조문이 위로 올라오는 일이 생길 수 있습니다. 이런 결과는 Step 8의 "예상과 달랐던 점"에 그대로 적을 거리입니다.

#### 트랙 3 : 알아차리기 힘든 실패 3종 진단

로컬 모델 운영에서 가장 위험한 버그는 에러를 내지 않고 성능만 조용히 떨어뜨립니다.

1. **토큰 상한 초과 절단** : 512 토큰 모델에 긴 청크를 넣으면 뒷부분이 조용히 사라집니다
2. **prefix 누락** : Step 3에서 이미 확인했습니다 (`RESULTS`의 E5 두 줄 비교)
3. **정규화 누락** : 정규화하지 않은 벡터에 내적을 쓰면 코사인 유사도가 아니게 됩니다

In [44]:
# 실패 1-a : 레슨의 가상 긴 조항 (핵심 정보를 맨 끝에 숨김)
# 레슨에서는 "제12조"라는 이름이었지만 근로기준법에도 제12조가 있으므로 이름을 바꿨습니다
items = "\n".join(
    f"{i}. 제{i}호에 관한 세부 시행 절차는 담당 부서의 내부 지침에 따르며, "
    f"필요한 경우 관련 부서와 협의하여 조정할 수 있다."
    for i in range(1, 26)
)
long_clause = (
    "[가상 조항] 별표: 세부 시행 절차\n"
    + items
    + "\n26. 재해 발생 시 비상연락망은 총무팀장이 관리하며, 매 분기 첫째 주에 갱신한다."
)
print(f"글자 수     : {len(long_clause)}")
print(f"E5 토큰 수  : {len(tok_e5.encode(long_clause))}")
print(f"BGE 토큰 수 : {len(tok_bge.encode(long_clause))}")

글자 수     : 1825
E5 토큰 수  : 944
BGE 토큰 수 : 944


In [45]:
def rank_of(vecs, q_vec, target):
    order = list(np.argsort(vecs @ q_vec)[::-1])
    return order.index(target) + 1


test_corpus = chunk_texts + [long_clause]
gold_idx = len(test_corpus) - 1
query = "비상연락망은 누가 관리해?"

# E5 (prefix 규칙 준수)
e5_test_vecs = np.array(e5_embeddings.embed_documents([f"passage: {t}" for t in test_corpus]))
e5_rank = rank_of(e5_test_vecs, np.array(e5_embeddings.embed_query(f"query: {query}")), gold_idx)

# BGE-M3
bge_test_vecs = np.array(bge_embeddings.embed_documents(test_corpus))
bge_rank = rank_of(bge_test_vecs, np.array(bge_embeddings.embed_query(query)), gold_idx)

print(f"코퍼스 {len(test_corpus)}개 중")
print(f"E5  에서 긴 조항의 순위 : {e5_rank}위")
print(f"BGE 에서 긴 조항의 순위 : {bge_rank}위")

코퍼스 132개 중
E5  에서 긴 조항의 순위 : 102위
BGE 에서 긴 조항의 순위 : 1위


In [46]:
# 실패 1-b : 실제 근로기준법 조문으로 확인합니다
# 제74조(임산부의 보호)는 E5 기준 512 토큰이 넘는 긴 조문이고, 출퇴근 시각 변경(제9항)은 조문 맨 뒤쪽에 있습니다
q = "임신한 직원이 출퇴근 시간을 바꿔 달라고 하면 회사가 들어줘야 해?"
gold = id2idx["제74조"]
print(f"제74조 토큰 수 : E5 {len(tok_e5.encode(chunk_texts[gold]))} / BGE {len(tok_bge.encode(chunk_texts[gold]))}")

e5_rank_real = rank_of(e5_doc_vecs, np.array(e5_embeddings.embed_query(f"query: {q}")), gold)
bge_rank_real = rank_of(bge_doc_vecs, np.array(bge_embeddings.embed_query(q)), gold)
print(f"E5  에서 제74조의 순위 : {e5_rank_real}위")
print(f"BGE 에서 제74조의 순위 : {bge_rank_real}위")

제74조 토큰 수 : E5 706 / BGE 706
E5  에서 제74조의 순위 : 1위
BGE 에서 제74조의 순위 : 2위


실제 조문은 가상 조항만큼 결과가 극적이지 않을 수 있습니다. 제74조는 앞부분도 "임신 중의 여성"에 관한 내용이라, 뒷부분이 잘려도 앞부분만으로 검색될 수 있기 때문입니다. **절단이 항상 검색 실패로 이어지지는 않지만, 뒷부분에만 있는 정보는 임베딩에 반영되지 않는다**는 점이 핵심입니다.

In [47]:
# 실패 3 : 정규화 누락
# BGE-M3의 sentence-transformers 파이프라인에는 마지막에 Normalize 층이 있어서
# normalize_embeddings=False 로 호출해도 단위 벡터가 나옵니다. 그래서 Normalize 층을 뺀 모델로 원시 벡터를 만듭니다
from sentence_transformers import SentenceTransformer

full_model = SentenceTransformer("BAAI/bge-m3", device="cpu")   # 데모용 별도 로드는 CPU로 (VRAM 절약)
print(full_model)
raw_model = SentenceTransformer(modules=[full_model[0], full_model[1]], device="cpu")   # Normalize 층 제거

raw_vecs = raw_model.encode(chunk_texts, normalize_embeddings=False)
norms = np.linalg.norm(raw_vecs, axis=1)
print(f"벡터 길이 최소 : {norms.min():.3f}")
print(f"벡터 길이 최대 : {norms.max():.3f}")
print("정규화된 경우라면 모두 1.000이어야 합니다")

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)
벡터 길이 최소 : 24.696
벡터 길이 최대 : 27.154
정규화된 경우라면 모두 1.000이어야 합니다


In [48]:
# 정규화 여부에 따라 순위가 달라지는지 확인합니다. 조문 길이 편차가 큰 문서라 레슨보다 차이가 날 수 있습니다
changed = 0
for q, gold in sem_eval_set:
    raw_q = raw_model.encode(q, normalize_embeddings=False)
    norm_q = np.array(bge_embeddings.embed_query(q))
    raw_top = [article_ids[i] for i in np.argsort(raw_vecs @ raw_q)[::-1][:3]]
    norm_top = [article_ids[i] for i in np.argsort(bge_doc_vecs @ norm_q)[::-1][:3]]
    if raw_top != norm_top:
        changed += 1
        print(f"Q : {q}")
        print(f"  정규화 안 함 top3 : {raw_top}")
        print(f"  정규화 함   top3 : {norm_top}")
print(f"\n의미 질문 {len(sem_eval_set)}개 중 top3 가 달라진 질문 : {changed}개")

record("진단 : 정규화 누락 (원시 벡터 내적)",
       lambda query, k=3: list(np.argsort(raw_vecs @ raw_model.encode(query, normalize_embeddings=False))[::-1][:k]))

del full_model, raw_model
gc.collect()

Q : 1년 일하면 연차 며칠 받을 수 있어?
  정규화 안 함 top3 : ['제60조', '제84조', '제71조']
  정규화 함   top3 : ['제60조', '제84조', '제73조']
Q : 4시간 일하면 쉬는 시간은 얼마나 줘야 해?
  정규화 안 함 top3 : ['제54조', '제59조', '제50조']
  정규화 함   top3 : ['제54조', '제50조', '제52조']
Q : 그만두면 밀린 월급은 언제까지 받아야 해?
  정규화 안 함 top3 : ['제36조', '제37조', '제61조']
  정규화 함   top3 : ['제36조', '제37조', '제60조']
Q : 상사가 지위를 이용해서 괴롭히는 것도 법에 걸려?
  정규화 안 함 top3 : ['제76조의2', '제116조', '제104조']
  정규화 함   top3 : ['제76조의2', '제116조', '제76조의3']
Q : 몇 살부터 알바할 수 있어?
  정규화 안 함 top3 : ['제64조', '제70조', '제68조']
  정규화 함   top3 : ['제64조', '제70조', '제69조']
Q : 출산휴가는 며칠 쓸 수 있어?
  정규화 안 함 top3 : ['제74조', '제60조', '제71조']
  정규화 함   top3 : ['제74조', '제60조', '제70조']
Q : 일주일에 기본으로 몇 시간까지 일할 수 있어?
  정규화 안 함 top3 : ['제50조', '제51조', '제51조의2']
  정규화 함   top3 : ['제69조', '제50조', '제51조']
Q : 그만두면 위약금 물어내라는 계약서 써도 돼?
  정규화 안 함 top3 : ['제20조', '제19조', '제26조']
  정규화 함   top3 : ['제19조', '제20조', '제30조']
Q : 근로계약서에 꼭 들어가야 하는 내용이 뭐야?
  정규화 안 함 top3 : ['제17조', '제42조', '제93조']
  정규화 함   top3 : ['제17조', '제93조

7186

세 가지 실패 모두 오류를 내지 않았습니다. 여기서 얻어야 할 습관은 세 가지입니다.

- 새 임베딩 모델을 쓸 때 **모델 카드에서 최대 입력 길이와 prefix 규칙을 먼저 확인**합니다
- 인덱싱 후 **청크의 토큰 수 분포를 한 번은 출력**합니다 (Step 2의 표)
- **벡터 길이가 1인지 확인**합니다 (`np.linalg.norm(vecs, axis=1)`)

#### 트랙 4 : 부모-자식 청킹

**검색은 작은 단위로, 생성은 큰 단위로** 합니다. 레슨은 "다."로 문장을 나눴지만, 법령은 항(①②…)과 호(1. 2. …)가 **줄 단위**로 나뉘어 있으므로 줄을 자식 단위로 씁니다. 10자 이하의 짧은 줄(예: "1. 임금")은 단독으로는 의미가 약해 제외하고, 줄이 하나도 남지 않는 조문은 조문 전체를 자식 하나로 씁니다.

In [49]:
children = []
for parent_id, c in enumerate(chunks):
    parent_text = c.page_content
    lines = [l.strip() for l in parent_text.split("\n") if len(l.strip()) > 10]
    for s in (lines or [parent_text.strip()]):
        children.append({
            "text": s,
            "parent_id": parent_id,
            "parent_text": parent_text,
            "조항": c.metadata.get("조항", ""),
        })

print(f"부모 청크 : {len(chunks)}개")
print(f"자식 청크 : {len(children)}개")
print(f"\n예시 자식 : {children[5]['text'][:80]}")

# 자식 청크를 별도 컬렉션에 인덱싱합니다
CHILD_COLLECTION = "law_children"
child_vecs = np.array(bge_embeddings.embed_documents([c["text"] for c in children]))

if client.collection_exists(CHILD_COLLECTION):
    client.delete_collection(CHILD_COLLECTION)
client.create_collection(
    collection_name=CHILD_COLLECTION,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)
client.upsert(
    collection_name=CHILD_COLLECTION,
    points=[PointStruct(id=i, vector=child_vecs[i].tolist(), payload=children[i]) for i in range(len(children))],
)
print(f"{len(children)}개 자식 청크 저장 완료")

부모 청크 : 131개
자식 청크 : 409개

예시 자식 : 4. “근로계약”이란 근로자가 사용자에게 근로를 제공하고 사용자는 이에 대하여 임금을 지급하는 것을 목적으로 체결된 계약을 말한다.
409개 자식 청크 저장 완료


In [50]:
# 실습 : 자식으로 검색하고 부모로 확장합니다. 같은 부모는 한 번만, 검색 순위는 유지합니다
def retrieve_parent(query, child_top_k=10, max_parents=3):
    hits = client.query_points(
        collection_name=CHILD_COLLECTION,
        query=bge_embeddings.embed_query(query),
        limit=child_top_k,
    ).points

    parents, seen = [], set()
    for h in hits:
        pid = h.payload["parent_id"]
        if pid in seen:
            continue
        seen.add(pid)
        parents.append({
            "parent_id": pid,
            "조항": h.payload["조항"],
            "parent_text": h.payload["parent_text"],
            "matched_child": h.payload["text"],
            "유사도": h.score,
        })
    return parents[:max_parents]


for p in retrieve_parent("1년 일하면 연차 며칠 받을 수 있어?"):
    print(f"[{p['조항']}] {p['parent_text'][:60]}...")

[제60조(연차 유급휴가)] ① 사용자는 1년간 80퍼센트 이상 출근한 근로자에게 15일의 유급휴가를 주어야 한다.
② 사용자는 계속하여...
[제84조(일시보상)] 제78조에 따라 보상을 받는 근로자가 요양을 시작한 지 2년이 지나도 부상 또는 질병이 완치되지 아니하는 경...
[제74조(임산부의 보호)] ① 사용자는 임신 중의 여성에게 출산 전과 출산 후를 통하여 90일(미숙아를 출산한 경우에는 100일, 한 ...


In [51]:
# 기존 조항 단위 검색과 비교합니다
q = "그만두면 밀린 월급은 언제까지 받아야 해?"

print("=== 조항 단위 검색 (Step 6 방식) ===")
for c in retrieve(q, 3):
    print(f"[{c['조항']}] 유사도 {c['유사도']:.3f}")

print("\n=== 자식 검색 + 부모 확장 ===")
for p in retrieve_parent(q):
    print(f"[{p['조항']}] 유사도 {p['유사도']:.3f}")
    print(f"  걸린 문장 : {p['matched_child'][:50]}...")

record("부모-자식 청킹 : 항·호 단위 자식 검색 → 조문 확장",
       lambda query, k=3: [p["parent_id"] for p in retrieve_parent(query, child_top_k=max(10, k * 3), max_parents=k)])

=== 조항 단위 검색 (Step 6 방식) ===
[제36조(금품 청산)] 유사도 0.624
[제37조(미지급 임금에 대한 지연이자)] 유사도 0.620
[제60조(연차 유급휴가)] 유사도 0.616

=== 자식 검색 + 부모 확장 ===
[제36조(금품 청산)] 유사도 0.624
  걸린 문장 : 사용자는 근로자가 사망 또는 퇴직한 경우에는 그 지급 사유가 발생한 때부터 14일 이내에 ...
[제38조(임금채권의 우선변제)] 유사도 0.621
  걸린 문장 : 1. 최종 3개월분의 임금...
[제52조(선택적 근로시간제)] 유사도 0.612
  걸린 문장 : 1. 근로일 종료 후 다음 근로일 시작 전까지 근로자에게 연속하여 11시간 이상의 휴식 시...
[기록] 부모-자식 청킹 : 항·호 단위 자식 검색 → 조문 확장
       Hit@3 전체 70% / 의미 100% / 식별자 0%,  26 ms/질문


In [52]:
# 부모 확장 방식으로 답변을 만들어 조항 단위 방식과 비교합니다
# 조건부 규정("원칙은 이렇지만, 이런 경우는 예외")에서 차이가 드러나기 쉽습니다
def rag_answer_parent(query: str, max_parents: int = 3) -> str:
    parents = retrieve_parent(query, max_parents=max_parents)
    context_text = "\n\n".join(f"[{p['조항']}]\n{p['parent_text']}" for p in parents)
    user_prompt = f"다음은 근로기준법에서 검색된 조문입니다.\n\n{context_text}\n\n질문 : {query}"
    return generate(SYSTEM_PROMPT, user_prompt)


q = "회사가 내가 신청한 연차 날짜를 바꾸라고 할 수 있어?"
print("=== 조항 단위 검색 ===")
print(rag_answer(q))
print("\n=== 자식 검색 + 부모 확장 ===")
print(rag_answer_parent(q))

=== 조항 단위 검색 ===
회사는 근로자와 서면 합의에 따라 연차 유급휴가일을 갈음하여 특정 근로일에 근로자를 휴무시킬 수 있습니다. 따라서 회사가 근로자의 연차 날짜를 바꾸는 것은 가능하나, 이는 근로자와의 서면 합의가 필수적입니다.

근거 조항: 제62조

=== 자식 검색 + 부모 확장 ===
사용자는 근로자대표와의 서면 합의에 따라 연차 유급휴가일을 갈음하여 특정한 근로일에 근로자를 휴무시킬 수 있습니다. 따라서 회사는 근로자와의 합의가 있는 경우 연차 날짜를 변경할 수 있습니다 (제62조). 하지만 근로자와의 합의 없이 무단으로 변경하는 것은 가능하지 않습니다.


#### 트랙 5 : 로컬 리랭커 2단계 검색

1단계에서 바이 인코더(BGE-M3)로 후보를 추리고, 2단계에서 크로스 인코더(`bge-reranker-v2-m3`)로 후보만 정밀 채점합니다. 레슨 문서는 조항이 11개라 후보 20개가 사실상 전체였지만, 근로기준법은 131개라 **후보 선별이 실제로 의미를 가집니다.**

리랭커가 조번호를 볼 수 있는지에 따라 식별자 질문 결과가 달라질 수 있으므로, 두 컬렉션에서 각각 실험합니다.

- `labor_law` : 본문만 (기준선 컬렉션)
- `labor_law_full` : 장·조항 제목 포함 (트랙 2의 `full_texts`)

In [53]:
# 제목 포함 컬렉션을 만듭니다 (벡터는 트랙 2에서 만든 full_vecs 재사용)
FULL_COLLECTION = "labor_law_full"
if client.collection_exists(FULL_COLLECTION):
    client.delete_collection(FULL_COLLECTION)
client.create_collection(
    collection_name=FULL_COLLECTION,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)
client.upsert(
    collection_name=FULL_COLLECTION,
    points=[
        PointStruct(id=i, vector=full_vecs[i].tolist(),
                    payload={"text": full_texts[i], "조항": chunks[i].metadata.get("조항", ""), "조번호": article_ids[i]})
        for i in range(len(chunks))
    ],
)
print(f"{len(chunks)}개 포인트 저장 완료 ({FULL_COLLECTION})")

# 같은 벡터를 Qdrant로 검색해도 트랙 2의 numpy 결과와 같아야 합니다
record("청킹 변경 : 장·조항 제목 포함 (Qdrant)", lambda query, k=3: [h["id"] for h in retrieve(query, k, FULL_COLLECTION)])

131개 포인트 저장 완료 (labor_law_full)
[기록] 청킹 변경 : 장·조항 제목 포함 (Qdrant)
       Hit@3 전체 90% / 의미 93% / 식별자 83%,  25 ms/질문


In [54]:
# VRAM 확인. 리랭커는 약 2.2GB가 필요합니다
print(f"현재 VRAM : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# 여유가 부족하면 아래 두 줄로 LLM을 내렸다가 나중에 다시 올리세요
# del model
# torch.cuda.empty_cache()

현재 VRAM : 8.97 GB


In [55]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda")

# 동작 확인 : 같은 질문에 대한 두 조문의 점수를 비교합니다
pairs = [
    ("연차 유급휴가는 며칠인가요?", chunk_texts[id2idx["제60조"]]),   # 정답
    ("연차 유급휴가는 며칠인가요?", chunk_texts[id2idx["제50조"]]),   # 무관 (근로시간)
]
print(reranker.predict(pairs))

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

[0.636685   0.00313992]


In [56]:
# 실습 : 2단계 검색 함수
# 1단계에서 candidate_k 개를 검색하고, (질문, 문서) 쌍을 리랭커로 채점한 뒤 점수 순으로 top_k 개를 반환합니다
def two_stage_retrieve(query, candidate_k=20, top_k=3, collection=COLLECTION):
    hits = client.query_points(
        collection_name=collection,
        query=bge_embeddings.embed_query(query),
        limit=candidate_k,
    ).points

    pairs = [(query, h.payload["text"]) for h in hits]
    scores = reranker.predict(pairs)

    scored = [
        {"id": h.id, "조항": h.payload["조항"], "내용": h.payload["text"],
         "dense_score": h.score, "rerank_score": float(s)}
        for h, s in zip(hits, scores)
    ]
    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_k]


for r in two_stage_retrieve("1년 일하면 연차 며칠 받을 수 있어?"):
    print(f"[리랭커 점수 {r['rerank_score']:.3f} / 밀집 {r['dense_score']:.3f}] {r['조항']}")

[리랭커 점수 0.033 / 밀집 0.649] 제60조(연차 유급휴가)
[리랭커 점수 0.009 / 밀집 0.570] 제51조의3(근로한 기간이 단위기간보다 짧은 경우의 임금 정산)
[리랭커 점수 0.005 / 밀집 0.554] 제75조(육아 시간)


In [57]:
# 리랭킹이 순위를 얼마나 바꾸는지, 얼마나 느려지는지 측정합니다
def reranked_rank_fn(candidate_k, collection):
    return lambda query, k=3: [r["id"] for r in two_stage_retrieve(query, candidate_k, k, collection)]

record("리랭커 2단계 : 본문 컬렉션, 후보 20", reranked_rank_fn(20, COLLECTION))
record("리랭커 2단계 : 제목 포함 컬렉션, 후보 20", reranked_rank_fn(20, FULL_COLLECTION))

[기록] 리랭커 2단계 : 본문 컬렉션, 후보 20
       Hit@3 전체 70% / 의미 93% / 식별자 17%,  1818 ms/질문
[기록] 리랭커 2단계 : 제목 포함 컬렉션, 후보 20
       Hit@3 전체 95% / 의미 93% / 식별자 100%,  1817 ms/질문


In [58]:
# 후보 개수(candidate_k)를 바꿔가며 정확도와 지연 시간의 교환 관계를 기록합니다
for ck in [5, 50]:
    record(f"리랭커 2단계 : 제목 포함 컬렉션, 후보 {ck}", reranked_rank_fn(ck, FULL_COLLECTION))

[기록] 리랭커 2단계 : 제목 포함 컬렉션, 후보 5
       Hit@3 전체 85% / 의미 86% / 식별자 83%,  431 ms/질문
[기록] 리랭커 2단계 : 제목 포함 컬렉션, 후보 50
       Hit@3 전체 95% / 의미 93% / 식별자 100%,  3332 ms/질문


`ms/질문`을 함께 보세요. 후보가 많을수록 리랭커가 채점할 쌍이 늘어 느려집니다. 또 리랭커의 `max_length=512`도 **토큰 상한 절단**입니다. 긴 조문은 리랭커에서도 뒷부분이 잘린다는 점을 트랙 3과 연결해서 해석해 보세요.

## Step 8 : 결과 정리와 리포트

In [59]:
df = pd.DataFrame(RESULTS)

# 기준선을 맨 위로, 나머지는 실행 순서대로
df = pd.concat([df[df["실험"] == BASELINE_NAME], df[df["실험"] != BASELINE_NAME]], ignore_index=True)
base = df.loc[df["실험"] == BASELINE_NAME, "전체"].iloc[0]
df["기준선 대비"] = (df["전체"] - base).map(lambda x: f"{x * 100:+.1f}%p")

show = df.copy()
for col in ["전체", "의미", "식별자"]:
    show[col] = show[col].map(lambda x: f"{x:.0%}")
show["ms/질문"] = show["ms/질문"].map(lambda x: f"{x:.0f}")
show = show.rename(columns={"전체": "Hit@3 전체", "의미": "Hit@3 의미", "식별자": "Hit@3 식별자"})
print(show.to_string(index=False))

                                       실험 Hit@3 전체 Hit@3 의미 Hit@3 식별자 ms/질문  기준선 대비
   기준선 : 구조 기반 청킹 + BGE-M3 밀집 검색 (Qdrant)      70%     100%        0%    24  +0.0%p
임베딩 교체 : multilingual-e5-base (prefix 준수)      60%      86%        0%    14 -10.0%p
임베딩 교체 : multilingual-e5-base (prefix 누락)      60%      86%        0%    13 -10.0%p
             청킹 변경 : 장·조항 제목을 본문에 포함 (밀집)      90%      93%       83%    23 +20.0%p
                          BM25 (레슨 토크나이저)      55%      50%       67%     0 -15.0%p
                      BM25 (조의N 대응 토크나이저)      55%      50%       67%     0 -15.0%p
   하이브리드 : 밀집(제목 포함) + BM25(수정), RRF k=60      85%      86%       83%    24 +15.0%p
                   진단 : 정규화 누락 (원시 벡터 내적)      70%     100%        0%   432  +0.0%p
          부모-자식 청킹 : 항·호 단위 자식 검색 → 조문 확장      70%     100%        0%    26  +0.0%p
              청킹 변경 : 장·조항 제목 포함 (Qdrant)      90%      93%       83%    25 +20.0%p
                  리랭커 2단계 : 본문 컬렉션, 후보 20      70%      93%       17%  1818 

In [60]:
# 리포트에 붙여넣을 수 있도록 마크다운 표로도 출력합니다
cols = list(show.columns)
lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in show.itertuples(index=False)]
print("\n".join(lines))

| 실험 | Hit@3 전체 | Hit@3 의미 | Hit@3 식별자 | ms/질문 | 기준선 대비 |
|---|---|---|---|---|---|
| 기준선 : 구조 기반 청킹 + BGE-M3 밀집 검색 (Qdrant) | 70% | 100% | 0% | 24 | +0.0%p |
| 임베딩 교체 : multilingual-e5-base (prefix 준수) | 60% | 86% | 0% | 14 | -10.0%p |
| 임베딩 교체 : multilingual-e5-base (prefix 누락) | 60% | 86% | 0% | 13 | -10.0%p |
| 청킹 변경 : 장·조항 제목을 본문에 포함 (밀집) | 90% | 93% | 83% | 23 | +20.0%p |
| BM25 (레슨 토크나이저) | 55% | 50% | 67% | 0 | -15.0%p |
| BM25 (조의N 대응 토크나이저) | 55% | 50% | 67% | 0 | -15.0%p |
| 하이브리드 : 밀집(제목 포함) + BM25(수정), RRF k=60 | 85% | 86% | 83% | 24 | +15.0%p |
| 진단 : 정규화 누락 (원시 벡터 내적) | 70% | 100% | 0% | 432 | +0.0%p |
| 부모-자식 청킹 : 항·호 단위 자식 검색 → 조문 확장 | 70% | 100% | 0% | 26 | +0.0%p |
| 청킹 변경 : 장·조항 제목 포함 (Qdrant) | 90% | 93% | 83% | 25 | +20.0%p |
| 리랭커 2단계 : 본문 컬렉션, 후보 20 | 70% | 93% | 17% | 1818 | +0.0%p |
| 리랭커 2단계 : 제목 포함 컬렉션, 후보 20 | 95% | 93% | 100% | 1817 | +25.0%p |
| 리랭커 2단계 : 제목 포함 컬렉션, 후보 5 | 85% | 86% | 83% | 431 | +15.0%p |
| 리랭커 2단계 : 제목 포함 컬렉션, 후보 50 | 95% | 93% | 100% |

### 리포트

> ✍️ 표시가 있는 부분은 **실행 결과를 보고 직접 작성**합니다. 문서와 설계에 관한 부분은 미리 적어 두었습니다.

**1. 문서와 청킹 전략**

- 문서 : 근로기준법 본칙 131개 조문 (시행 2026. 8. 20. 기준, 시행예정 개정본·부칙·삭제 조문 제외)
- 청킹 전략 : 구조 기반(조 단위). 법령은 조가 의미 단위이자 인용 단위이고, 조문 길이 편차가 커서(최소 약 25자 ~ 최대 약 1,950자) 글자 수 기준 청킹은 짧은 조문을 섞거나 긴 조문을 자르게 됩니다
- 평가 질문 : 의미 질문 14개 + 식별자 질문 6개 (`제76조의2`, `제76조의3`처럼 헷갈리기 쉬운 식별자 포함)

**2. 기준선**

- ✍️ 기준선 Hit@3 (전체 / 의미 / 식별자) :
- 오프라인 검증 : Step 6 출력 참고 (소켓 차단 상태에서 답변 생성)

**3. 개선 실험 결과** : 위 마크다운 표를 붙여넣습니다

**4. 효과가 없었거나 오히려 나빠진 시도**

- ✍️

**5. 결과 해석** (문서의 성격 / 질문의 성격 / 모델의 특성 중 무엇이 원인인지)

- ✍️ 식별자 질문 : 본문에 조번호가 없는 기준선과 제목을 넣은 뒤의 차이, BM25가 인용 조문을 매치한 사례가 있었는지
- ✍️ 토크나이저 수정 : `제76조의2` / `제76조의3` 질문에서 레슨 토크나이저와 수정본의 차이
- ✍️ 리랭커 : 조문이 131개일 때 후보 5 / 20 / 50의 정확도와 지연 시간
- ✍️ 부모-자식 : 항·호 단위 자식이 의미 질문에서 도움이 됐는지, 식별자 질문에서는 어땠는지
- ✍️ 토큰 상한 : E5와 리랭커(512)가 긴 조문(제44조의3, 제74조 등)을 자르는 영향

**6. 실무에 적용한다면**

- ✍️ 법령·시행령·시행규칙 전체(수만 조문)로 늘리면 조번호가 법령마다 겹치므로(모든 법령에 제1조가 있음) 법령명 메타데이터와 필터가 필수가 됩니다. 이 외에 무엇이 병목이 될지 적어 보세요